# 06 Training Pipeline for Oil Price Prediction

In [1]:
import pandas as pd
import numpy as np
import s3fs
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib
import mlflow
import mlflow.sklearn

## Set up MLflow Experiment

In [2]:
import mlflow
mlflow.set_tracking_uri("http://mlflow.mlflow.svc.cluster.local")
mlflow.set_experiment("oilprice prediction")
mlflow.autolog()

2026/10/04 16:21:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


## Load Processed Features from S3

In [ ]:
# S3/Ceph object storage credentials - identical to the production pipelines.
# src/s3_credentials.py reads them from the environment: in Kubernetes the
# manifests inject them from secretKeyRef "s3-credentials" (aws_access_key_id /
# aws_secret_access_key) plus the RGW endpoint; locally they come from the repo
# .env (copy .env.example to .env). No kubectl, no base64, no credentials here.
import os
import sys

import s3fs

sys.path.insert(0, os.path.abspath("../src"))  # notebooks/ -> src/

from s3_credentials import get_s3_options

try:
    s3_options = get_s3_options()
except EnvironmentError as exc:
    # Local fallback: nothing is hard-coded in a notebook cell on purpose - a
    # placeholder key looks like a real one and hides the misconfiguration.
    print("Local run without S3 credentials in the environment.")
    print("Copy .env.example to .env in the repo root and set:")
    print("  AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY / AWS_ENDPOINT_URL")
    print("For a local MinIO: AWS_ENDPOINT_URL=http://localhost:8080")
    raise exc

try:
    if s3.exists('s3://oilprice-features/processed_features.parquet'):
        df = pd.read_parquet('s3://oilprice-features/processed_features.parquet', storage_options=s3_options)
    else:
        raise FileNotFoundError("Processed features not found in S3")
except Exception as e:
    print(f"Error loading data: {e}")


## Data Preprocessing

In [4]:
# Feature- & Label-Spalten (muensten mit 04_feature_engineering.ipynb uebereinstimmen)
AGG_FEATURES = [
    "price_mean_7d", "price_std_30d",
    "demand_mean_7d", "demand_mean_30d",
    "month", "season", "month_sin", "month_cos", "seasonal_index", "price_mean_dow_4w",
]
RT_FEATURES = ["current_price", "current_demand"]
FEATURES = AGG_FEATURES + RT_FEATURES
TARGET = "target_next_week_price"   # O der naechsten 7 Tage

# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Rows mit fehlenden Feature-/Label-Werten verwerfen (04 hat bereits dropna applied)
df = df.dropna(subset=FEATURES + [TARGET])

# Features vs. Ziel: nur FEATURES werden verwendet, die Labels (target_*) sind KEINE
# Features -> kein Leakage. RandomForest benoetigt keine Skalierung.
X = df[FEATURES].copy()
# Non-numerische Feature-Spalten (z.B. "season") als Integer-Codes kodieren:
# RandomForestRegressor (sklearn 1.9.1) hat keinen nativen Categorical-Support
# und konvertiert X strikt nach float -> Strings sind dort nicht erlaubt.
for col in X.columns:
    if not pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].astype("category").cat.codes
y = df[TARGET]

print(f"X: {X.shape} | y: {y.shape}")


Missing values per column:


date                       0
price_mean_7d              0
price_std_30d              0
demand_mean_7d             0
demand_mean_30d            0
month                      0
season                     0
month_sin                  0
month_cos                  0
seasonal_index             0
price_mean_dow_4w          0
current_price              0
current_demand             0
target_next_week_price     0
target_week_change         0
target_trend               0
target_next_month_price    0
dtype: int64
X: (8636, 12) | y: (8636,)


## Train-Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Train Model with MLflow Tracking

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import mlflow

# Identify categorical columns from the training data
categorical_cols = X_train.select_dtypes(include=["object", "category", "string"]).columns
numeric_cols = X_train.columns.difference(categorical_cols)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("numeric", "passthrough", numeric_cols),
    ]
)

model = make_pipeline(
    preprocessor,
    RandomForestRegressor(n_estimators=100, random_state=42),
)

with mlflow.start_run():
    # The model is logged exactly ONCE via mlflow.autolog() (enabled in the
    # "Set up MLflow Experiment" cell), including the training-dataset
    # association. Do NOT add an explicit mlflow.sklearn.log_model() call
    # here: it would log a second, duplicate model WITHOUT the dataset
    # association.
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)

    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    mlflow.log_metric("mse", mse)

    print(f"MSE: {mse}")


2026/10/04 16:21:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/user/git/oilprice/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


2026/10/04 16:21:50 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/user/git/oilprice/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


2026/10/04 16:21:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/10/04 16:21:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/10/04 16:21:54 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/user/git/oilprice/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


MSE: 2.4803171796407293
🏃 View run painted-flea-666 at: http://mlflow.mlflow.svc.cluster.local/#/experiments/2/runs/99985b577a8c40949be61ede1f269754
🧪 View experiment at: http://mlflow.mlflow.svc.cluster.local/#/experiments/2


## Evaluation Metrics

In [7]:
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 2.4803171796407293


### What does the MSE mean?

**Mean Squared Error** is the average of the squared differences between predicted and actual prices:

    MSE = (1 / n) * Sum over i of (y_true,i - y_pred,i)^2

* **Units:** squared price (CHF²/100L²) - the raw MSE is therefore hard to read directly. Take the square root: **RMSE = sqrt(MSE)** is back in CHF/100L and can be interpreted as the typical size of a prediction error.
* **Squaring penalises large errors strongly:** one large miss weighs more than several small ones, so the MSE is sensitive to outliers.
* **A low MSE is only meaningful in context:** it has to be judged against the price level and against a naive baseline (always predicting the training mean). The next cell shows whether the model actually beats that baseline.


In [8]:
# Naive baseline: always predict the training mean (no model at all)
baseline_mse = mean_squared_error(y_test, np.full(len(y_test), float(y_train.mean())))

print(f"MSE  (model):         {mse:.4f}  ->  RMSE {mse ** 0.5:.4f} CHF/100L")
print(f"MSE  (mean baseline): {baseline_mse:.4f}  ->  RMSE {baseline_mse ** 0.5:.4f} CHF/100L")
print(f"The model reduces the squared error by {(1 - mse / baseline_mse) * 100:.1f}% compared to the mean baseline.")


MSE  (model):         2.4803  ->  RMSE 1.5749 CHF/100L
MSE  (mean baseline): 677.9402  ->  RMSE 26.0373 CHF/100L
The model reduces the squared error by 99.6% compared to the mean baseline.


## Future Predictions

The model's target is `target_next_week_price` - the **average price of the 7 days after** the given date. Every number below is therefore a "7-day average price" anchored at that date.

For dates that are not in the dataset, the feature row is built like this:

* **Calendar / seasonal features** (`month`, `season`, `month_sin`, `month_cos`, `seasonal_index`) are computed for the target date. The `seasonal_index` is a fixed per-month value learned from the history (1.0 = yearly average), so it is known for any future month - that is what makes a far-horizon prediction like March 2027 possible at all.
* **Level features** (`price_mean_7d`, `price_std_30d`, `demand_mean_7d`, `demand_mean_30d`, `price_mean_dow_4w`, `current_price`, `current_demand`) are **carried forward from the most recent observation** - a "no-trend" assumption: price and demand stay at their last observed level. This is the standard way to apply this model to future dates and is the main source of uncertainty, especially for long horizons.


In [9]:
# Baseline: most recent observation in the dataset
last = df.loc[df["date"].idxmax()]
last_date = pd.Timestamp(last["date"])
print(f"Last observation date: {last_date.date()}")

# Category mapping EXACTLY as used during training (order of first appearance).
# X["season"] already holds the training codes, so we can verify the mapping.
cat_code_map = {}
for col in FEATURES:
    if not pd.api.types.is_numeric_dtype(df[col]):
        categories = df[col].astype("category").cat.categories
        cat_code_map[col] = {v: i for i, v in enumerate(categories)}
        assert (df[col].map(cat_code_map[col]) == X[col]).all(), f"category mapping mismatch for {col}"

def future_features(date: pd.Timestamp) -> pd.DataFrame:
    """Feature row for a future date: calendar/seasonal features are computed
    for the date itself; level/rolling features are carried forward from the
    most recent observation (no-trend assumption)."""
    m = date.month
    season = {
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Frühling", 4: "Frühling", 5: "Frühling",
        6: "Sommer", 7: "Sommer", 8: "Sommer",
        9: "Herbst", 10: "Herbst", 11: "Herbst",
    }[m]
    row = {
        "price_mean_7d": last["price_mean_7d"],
        "price_std_30d": last["price_std_30d"],
        "demand_mean_7d": last["demand_mean_7d"],
        "demand_mean_30d": last["demand_mean_30d"],
        "month": m,
        "season": cat_code_map["season"][season],
        "month_sin": np.sin(2 * np.pi * m / 12),
        "month_cos": np.cos(2 * np.pi * m / 12),
        "seasonal_index": float(df.loc[df["month"] == m, "seasonal_index"].iloc[0]),
        "price_mean_dow_4w": last["price_mean_dow_4w"],
        "current_price": last["current_price"],
        "current_demand": last["current_demand"],
    }
    return pd.DataFrame([row])[FEATURES]

def predict_at(date: pd.Timestamp) -> float:
    """Predicted 7-day average price (CHF/100L) after `date`."""
    return float(model.predict(future_features(date))[0])

# --- short-term: next day / next week / next month ----------------------------
horizons = {
    "next day":   last_date + pd.Timedelta(days=1),
    "next week":  last_date + pd.Timedelta(days=7),
    "next month": last_date + pd.DateOffset(months=1),
}
print("Short-term forecasts (7-day average price, CHF/100L):")
for label, d in horizons.items():
    print(f"  {label:<12} from {d.date()}  ->  {predict_at(d):.2f}")

# --- long-term: March 2027 ------------------------------------------------------
# Weekly grid across March 2027; averaging the weekly forecasts approximates
# the expected level for the whole month.
march_dates = [pd.Timestamp("2027-03-01") + pd.Timedelta(weeks=w) for w in range(5)]
march_forecasts = [(d, predict_at(d)) for d in march_dates]
print()
print("March 2027, weekly forecasts (7-day average price, CHF/100L):")
for d, p in march_forecasts:
    print(f"  {d.date()}  ->  {p:.2f}")
print(f"  average over March 2027:  {np.mean([p for _, p in march_forecasts]):.2f} CHF/100L")


2026/10/04 16:21:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/user/git/oilprice/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Last observation date: 2026-09-25
Short-term forecasts (7-day average price, CHF/100L):
  next day     from 2026-09-26  ->  167.09


2026/10/04 16:21:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/user/git/oilprice/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


  next week    from 2026-10-02  ->  167.02


2026/10/04 16:21:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/user/git/oilprice/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


...

2026/10/04 16:21:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/user/git/oilprice/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


2026/10/04 16:21:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/user/git/oilprice/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


2026/10/04 16:21:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/user/git/oilprice/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


2026/10/04 16:21:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/user/git/oilprice/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."



March 2027, weekly forecasts (7-day average price, CHF/100L):
  2027-03-01  ->  167.07
  2027-03-08  ->  167.07
  2027-03-15  ->  167.07
  2027-03-22  ->  167.07
  2027-03-29  ->  167.07
  average over March 2027:  167.07 CHF/100L
